# Batch 1 — Preprocessing (Baseline)

**Role in the experiment plan:** this is the **baseline** batch. Per the
project plan, Batch 1 uses:
- all reasonable features except `Taxpayer_ID` (minimal feature removal —
  multicollinearity/VIF-driven removal is Batch 2's job)
- correct categorical encoding
- model-specific scaling (scale-sensitive models only)
- **no imbalance handling** (no class weights, no sampling) — this batch
  exists to measure what "doing nothing about imbalance" looks like, so
  Batch 2's imbalance experiments have something concrete to compare against

This notebook produces one **fixed, stratified train/validation/test split**
that will be reused (not re-randomized) by every later batch, and builds the
model-specific feature matrices that `training.ipynb` consumes.

Recap from `EDA.ipynb`: the dataset is ~10.7% fraud / ~89.3% not fraud, and
the top predictors are `Tax_Gap`, `Previous_Violations`, `Invoice_Mismatch`,
`Missing_Documents`, `Cash_Transactions_Percentage`, plus `Business_Type` /
`Industry_Risk`. Several financial columns are near-exact mathematical
duplicates of each other (e.g. `VAT_Collected` ≈ 14% × `Annual_Revenue`) —
we deliberately keep all of them in Batch 1 and revisit removal in Batch 2
once we have a baseline to measure the impact against.


In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths (project structure)

In [2]:
# This notebook lives in Models_Batch_1/, one level below the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_1' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
SPLITS_DIR = DATA_DIR / 'splits'
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_1'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / 'models').mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)


PROJECT_ROOT: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model
DATA_DIR: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model\data
ARTIFACTS_DIR: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model\Models_Batch_1\artifacts


## 2. Load Cleaned Data

Output of `notebooks/Cleaning.ipynb`.

In [3]:
df = pd.read_csv(DATA_DIR / 'tax_fraud_dataset_cleaned.csv')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Fraud rate: {df['Fraud'].mean():.4f}")
df.head()


Shape: 50000 rows, 25 columns
Missing values: 0
Fraud rate: 0.1072


,Taxpayer_ID,Business_Type,Region,Years_in_Business,Employee_Count,Annual_Revenue,Annual_Expenses,Net_Profit,Taxable_Income,Expected_Tax,Declared_Tax,VAT_Collected,VAT_Paid,Previous_Audits,Previous_Violations,Late_Payments,Industry_Risk,Cash_Transactions_Percentage,Missing_Documents,Invoice_Mismatch,Expense_Ratio,Profit_Margin,Revenue_per_Employee,Tax_Gap,Fraud
0,100000,Restaurant,Cairo,10,9,1892448.36,839511.26,1052937.10,919634.40,206917.74,200930.13,264942.77,117531.58,2,0,0,Medium,58.87,0,0,0.444,0.556,210272.04,5987.61,0
1,100001,Pharmacy,Giza,0,17,2129742.33,923661.40,1206080.92,1078021.30,242554.79,234562.43,298163.93,129312.60,1,0,1,Low,5.98,0,0,0.434,0.566,125278.96,7992.36,0
2,100002,Restaurant,Sharqia,4,8,789223.93,334017.67,455206.26,420736.46,94665.70,81056.42,110491.35,46762.47,1,0,0,Medium,69.98,4,1,0.423,0.577,98652.99,13609.28,1
3,100003,Manufacturing,Cairo,7,8,427211.55,361951.79,65259.77,58963.05,13266.69,12816.14,59809.62,50673.25,2,0,0,Medium,42.56,1,0,0.847,0.153,53401.44,450.54,0
4,100004,Import/Export,Sharqia,3,8,1516452.68,831833.16,684619.52,588453.59,132402.06,130966.03,212303.38,116456.64,0,1,2,High,39.09,1,0,0.549,0.451,189556.58,1436.03,0


## 3. Fixed Stratified Train / Validation / Test Split (70 / 15 / 15)

This split is created **once** and saved as Taxpayer_ID lists under
`data/splits/`. Every batch's `processing.ipynb` checks for these files
first and reuses them instead of re-splitting — this guarantees Batch 1,
2, and 3 are always evaluated on the *exact same* validation and test rows,
so differences in results reflect the modeling changes, not a different
random split.


In [4]:
train_ids_path = SPLITS_DIR / 'train_ids.csv'
val_ids_path = SPLITS_DIR / 'val_ids.csv'
test_ids_path = SPLITS_DIR / 'test_ids.csv'

if train_ids_path.exists() and val_ids_path.exists() and test_ids_path.exists():
    print("Fixed split already exists — reusing it (not re-randomizing).")
    train_ids = pd.read_csv(train_ids_path)['Taxpayer_ID']
    val_ids = pd.read_csv(val_ids_path)['Taxpayer_ID']
    test_ids = pd.read_csv(test_ids_path)['Taxpayer_ID']
else:
    print("No existing split found — creating it now (this only happens once).")
    # Step 1: 70% train vs 30% temp (stratified by Fraud)
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['Fraud'], random_state=RANDOM_STATE
    )
    # Step 2: split the 30% temp into 15% val / 15% test (i.e. 50/50 of temp)
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['Fraud'], random_state=RANDOM_STATE
    )

    train_ids, val_ids, test_ids = train_df['Taxpayer_ID'], val_df['Taxpayer_ID'], test_df['Taxpayer_ID']

    train_ids.to_frame().to_csv(train_ids_path, index=False)
    val_ids.to_frame().to_csv(val_ids_path, index=False)
    test_ids.to_frame().to_csv(test_ids_path, index=False)
    print("Saved fixed split to data/splits/")

train_df = df[df['Taxpayer_ID'].isin(train_ids)].reset_index(drop=True)
val_df = df[df['Taxpayer_ID'].isin(val_ids)].reset_index(drop=True)
test_df = df[df['Taxpayer_ID'].isin(test_ids)].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} rows ({len(train_df)/len(df)*100:.1f}%), fraud rate = {train_df['Fraud'].mean():.4f}")
print(f"Val:   {len(val_df)} rows ({len(val_df)/len(df)*100:.1f}%), fraud rate = {val_df['Fraud'].mean():.4f}")
print(f"Test:  {len(test_df)} rows ({len(test_df)/len(df)*100:.1f}%), fraud rate = {test_df['Fraud'].mean():.4f}")

assert set(train_ids) & set(val_ids) == set()
assert set(train_ids) & set(test_ids) == set()
assert set(val_ids) & set(test_ids) == set()
print("\nNo overlap between splits — confirmed.")


No existing split found — creating it now (this only happens once).
Saved fixed split to data/splits/

Train: 35000 rows (70.0%), fraud rate = 0.1072
Val:   7500 rows (15.0%), fraud rate = 0.1072
Test:  7500 rows (15.0%), fraud rate = 0.1072

No overlap between splits — confirmed.


**The TEST set is now locked.** From here on, `test_df` is only touched
once, at the very end of the whole experiment (Batch 1/2/3), to evaluate the
final selected model(s). All batch-to-batch decisions use train (CV) and
validation performance only.


## 4. Feature / Target Definition

Drop `Taxpayer_ID` (identifier, not a feature). Keep every other column for
this baseline batch.


In [5]:
TARGET = 'Fraud'
DROP_COLS = ['Taxpayer_ID']

onehot_cols = ['Business_Type', 'Region']
ordinal_col = 'Industry_Risk'
ordinal_order = ['Low', 'Medium', 'High']  # Low < Medium < High

feature_cols = [c for c in df.columns if c not in DROP_COLS + [TARGET]]
numeric_cols = [c for c in feature_cols if c not in onehot_cols + [ordinal_col]]

print(f"Total feature columns: {len(feature_cols)}")
print(f"One-hot columns: {onehot_cols}")
print(f"Ordinal column: {ordinal_col} (order: {ordinal_order})")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

X_train_raw = train_df[feature_cols].copy()
X_val_raw = val_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()
y_train = train_df[TARGET].to_numpy()
y_val = val_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()


Total feature columns: 23
One-hot columns: ['Business_Type', 'Region']
Ordinal column: Industry_Risk (order: ['Low', 'Medium', 'High'])
Numeric columns (20): ['Years_in_Business', 'Employee_Count', 'Annual_Revenue', 'Annual_Expenses', 'Net_Profit', 'Taxable_Income', 'Expected_Tax', 'Declared_Tax', 'VAT_Collected', 'VAT_Paid', 'Previous_Audits', 'Previous_Violations', 'Late_Payments', 'Cash_Transactions_Percentage', 'Missing_Documents', 'Invoice_Mismatch', 'Expense_Ratio', 'Profit_Margin', 'Revenue_per_Employee', 'Tax_Gap']


## 5. Encoding

- `Business_Type`, `Region` → **One-Hot Encoding** (nominal, no natural order)
- `Industry_Risk` → **Ordinal Encoding** with explicit order `Low < Medium < High`
  (an arbitrary `LabelEncoder` would assign alphabetical order —
  `High=0, Low=1, Medium=2` — which would be mathematically wrong for a
  model that treats the number as ordered)

All encoders are **fit on the training set only**, then used to transform
validation and test — this avoids leaking information about the val/test
distributions into preprocessing.


In [6]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
onehot_encoder.fit(X_train_raw[onehot_cols])

ordinal_encoder = OrdinalEncoder(categories=[ordinal_order])
ordinal_encoder.fit(X_train_raw[[ordinal_col]])

onehot_feature_names = list(onehot_encoder.get_feature_names_out(onehot_cols))
print("One-hot expands to:", onehot_feature_names)
print("Ordinal mapping:", dict(zip(ordinal_order, range(len(ordinal_order)))))


One-hot expands to: ['Business_Type_Construction', 'Business_Type_Education', 'Business_Type_Healthcare', 'Business_Type_IT', 'Business_Type_Import/Export', 'Business_Type_Manufacturing', 'Business_Type_Pharmacy', 'Business_Type_Restaurant', 'Business_Type_Retail', 'Business_Type_Unknown', 'Region_Alex', 'Region_Assiut', 'Region_Cairo', 'Region_Dakahlia', 'Region_Giza', 'Region_Monufia', 'Region_Sharqia', 'Region_Sohag', 'Region_Unknown']
Ordinal mapping: {'Low': 0, 'Medium': 1, 'High': 2}


In [7]:
def encode_unscaled(X):
    """Encode categoricals, leave numeric columns as-is (for tree/boosting models + Naive Bayes)."""
    onehot_part = pd.DataFrame(
        onehot_encoder.transform(X[onehot_cols]),
        columns=onehot_feature_names, index=X.index
    )
    ordinal_part = pd.DataFrame(
        ordinal_encoder.transform(X[[ordinal_col]]),
        columns=[ordinal_col + '_ord'], index=X.index
    )
    numeric_part = X[numeric_cols].reset_index(drop=True)
    onehot_part = onehot_part.reset_index(drop=True)
    ordinal_part = ordinal_part.reset_index(drop=True)
    return pd.concat([numeric_part, ordinal_part, onehot_part], axis=1)

X_train_tree = encode_unscaled(X_train_raw)
X_val_tree = encode_unscaled(X_val_raw)
X_test_tree = encode_unscaled(X_test_raw)

tree_feature_names = X_train_tree.columns.tolist()
print(f"Encoded (unscaled) feature matrix: {X_train_tree.shape[1]} columns")
X_train_tree.head()


Encoded (unscaled) feature matrix: 40 columns


,Years_in_Business,Employee_Count,Annual_Revenue,Annual_Expenses,Net_Profit,Taxable_Income,Expected_Tax,Declared_Tax,VAT_Collected,VAT_Paid,Previous_Audits,Previous_Violations,Late_Payments,Cash_Transactions_Percentage,Missing_Documents,Invoice_Mismatch,Expense_Ratio,Profit_Margin,Revenue_per_Employee,Tax_Gap,Industry_Risk_ord,Business_Type_Construction,Business_Type_Education,Business_Type_Healthcare,Business_Type_IT,Business_Type_Import/Export,Business_Type_Manufacturing,Business_Type_Pharmacy,Business_Type_Restaurant,Business_Type_Retail,Business_Type_Unknown,Region_Alex,Region_Assiut,Region_Cairo,Region_Dakahlia,Region_Giza,Region_Monufia,Region_Sharqia,Region_Sohag,Region_Unknown
0,10,9,1892448.36,839511.26,1052937.10,919634.40,206917.74,200930.13,264942.77,117531.58,2,0,0,58.87,0,0,0.444,0.556,210272.04,5987.61,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,17,2129742.33,923661.40,1206080.92,1078021.30,242554.79,234562.43,298163.93,129312.60,1,0,1,5.98,0,0,0.434,0.566,125278.96,7992.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,8,1516452.68,831833.16,684619.52,588453.59,132402.06,130966.03,212303.38,116456.64,0,1,2,39.09,1,0,0.549,0.451,189556.58,1436.03,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,6,9,445465.32,291843.23,153622.09,151492.12,34085.73,33258.73,62365.14,40858.05,0,0,0,36.68,2,0,0.655,0.345,49496.15,827.00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10,9,1397865.77,1078888.70,318977.07,288528.93,64919.01,62407.75,195701.21,151044.42,1,0,0,7.54,1,0,0.772,0.228,155318.42,2511.26,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 6. Scaling for Scale-Sensitive Models

`StandardScaler` and `RobustScaler` are both fit on the training set's
**numeric columns only** (the one-hot / ordinal columns are already small
integers and don't need scaling). We keep both scaled versions — Batch 1's
training notebook will try each with Logistic Regression / SVM / MLP and
keep whichever scaler performs better by CV, directly answering the
"StandardScaler vs RobustScaler" comparison the project plan asks for.

Tree-based models (Decision Tree, Random Forest, XGBoost, LightGBM), Naive
Bayes (scale-invariant), and CatBoost (native categorical handling, see
below) do **not** get a scaled feature set — scaling would add preprocessing
cost with no benefit for these models.


In [8]:
def make_scaled(X_encoded, scaler):
    X_scaled = X_encoded.copy()
    X_scaled[numeric_cols] = scaler.transform(X_encoded[numeric_cols])
    return X_scaled

standard_scaler = StandardScaler().fit(X_train_tree[numeric_cols])
robust_scaler = RobustScaler().fit(X_train_tree[numeric_cols])

X_train_std = make_scaled(X_train_tree, standard_scaler)
X_val_std = make_scaled(X_val_tree, standard_scaler)
X_test_std = make_scaled(X_test_tree, standard_scaler)

X_train_rob = make_scaled(X_train_tree, robust_scaler)
X_val_rob = make_scaled(X_val_tree, robust_scaler)
X_test_rob = make_scaled(X_test_tree, robust_scaler)

print("Scaled feature matrices ready:", X_train_std.shape, X_train_rob.shape)


Scaled feature matrices ready: (35000, 40) (35000, 40)


## 7. Native Categorical Features for CatBoost

CatBoost can consume categorical columns directly (target-statistics
encoding internally) instead of one-hot, which avoids the dimensionality
blow-up from one-hot encoding and often performs better. We keep
`Business_Type` / `Region` as raw categories, and keep `Industry_Risk` as
the **ordinal-encoded integer** (not a raw category) so its Low<Medium<High
order is still explicit rather than treated as an unordered category.


In [9]:
def encode_for_catboost(X):
    X_cb = X.copy()
    X_cb[ordinal_col] = ordinal_encoder.transform(X_cb[[ordinal_col]])
    for c in onehot_cols:
        X_cb[c] = X_cb[c].astype(str)
    return X_cb

X_train_cb = encode_for_catboost(X_train_raw)
X_val_cb = encode_for_catboost(X_val_raw)
X_test_cb = encode_for_catboost(X_test_raw)

cat_feature_names = onehot_cols  # Business_Type, Region stay as native categories
cat_feature_indices = [X_train_cb.columns.get_loc(c) for c in cat_feature_names]

print("CatBoost feature matrix columns:", X_train_cb.columns.tolist())
print("Categorical feature indices for CatBoost:", cat_feature_indices, cat_feature_names)


CatBoost feature matrix columns: ['Business_Type', 'Region', 'Years_in_Business', 'Employee_Count', 'Annual_Revenue', 'Annual_Expenses', 'Net_Profit', 'Taxable_Income', 'Expected_Tax', 'Declared_Tax', 'VAT_Collected', 'VAT_Paid', 'Previous_Audits', 'Previous_Violations', 'Late_Payments', 'Industry_Risk', 'Cash_Transactions_Percentage', 'Missing_Documents', 'Invoice_Mismatch', 'Expense_Ratio', 'Profit_Margin', 'Revenue_per_Employee', 'Tax_Gap']
Categorical feature indices for CatBoost: [0, 1] ['Business_Type', 'Region']


## 8. Sanity Checks Before Saving

In [10]:
for name, X in [('tree', X_train_tree), ('std', X_train_std), ('rob', X_train_rob), ('catboost', X_train_cb)]:
    n_missing = X.isnull().sum().sum()
    print(f"{name:>10}: shape={X.shape}, missing_values={n_missing}")

assert X_train_tree.isnull().sum().sum() == 0
assert X_train_std.isnull().sum().sum() == 0
assert X_train_rob.isnull().sum().sum() == 0
assert X_train_cb.isnull().sum().sum() == 0
print("\nNo missing values in any feature matrix — OK.")


      tree: shape=(35000, 40), missing_values=0
       std: shape=(35000, 40), missing_values=0
       rob: shape=(35000, 40), missing_values=0
  catboost: shape=(35000, 23), missing_values=0

No missing values in any feature matrix — OK.


## 9. Save Artifacts for `training.ipynb`

In [11]:
artifact_bundle = {
    # unscaled encoded (tree-based models + Naive Bayes)
    'X_train_tree': X_train_tree, 'X_val_tree': X_val_tree, 'X_test_tree': X_test_tree,
    # StandardScaler version (Logistic Regression / SVM / MLP)
    'X_train_std': X_train_std, 'X_val_std': X_val_std, 'X_test_std': X_test_std,
    # RobustScaler version (compare against StandardScaler)
    'X_train_rob': X_train_rob, 'X_val_rob': X_val_rob, 'X_test_rob': X_test_rob,
    # native categorical version (CatBoost)
    'X_train_cb': X_train_cb, 'X_val_cb': X_val_cb, 'X_test_cb': X_test_cb,
    'cat_feature_indices': cat_feature_indices,
    # target
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    # metadata
    'tree_feature_names': tree_feature_names,
    'numeric_cols': numeric_cols,
    'onehot_cols': onehot_cols,
    'ordinal_col': ordinal_col,
    'ordinal_order': ordinal_order,
}

joblib.dump(artifact_bundle, ARTIFACTS_DIR / 'batch1_data.joblib')
joblib.dump(onehot_encoder, ARTIFACTS_DIR / 'onehot_encoder.joblib')
joblib.dump(ordinal_encoder, ARTIFACTS_DIR / 'ordinal_encoder.joblib')
joblib.dump(standard_scaler, ARTIFACTS_DIR / 'standard_scaler.joblib')
joblib.dump(robust_scaler, ARTIFACTS_DIR / 'robust_scaler.joblib')

print("Saved artifacts to:", ARTIFACTS_DIR)
for f in sorted(ARTIFACTS_DIR.glob('*.joblib')):
    print(" -", f.name)


Saved artifacts to: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model\Models_Batch_1\artifacts
 - batch1_data.joblib
 - onehot_encoder.joblib
 - ordinal_encoder.joblib
 - robust_scaler.joblib
 - standard_scaler.joblib


---
**Next:** `training.ipynb` loads `batch1_data.joblib`, trains and tunes all
9 models with cross-validation on the training set, and evaluates each on
the validation set.
